<a href="https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Each row represents one content item (content_hash_id) for one client (client_hash_id) on one calendar day (report_date). The analysis uses data from March 2026 (month = 2026-03), providing a single month of daily observations to evaluate and prioritize content refresh needs.

In [8]:
from getpass import getpass
import os
os.environ['HF_TOKEN'] = getpass('Paste your Hugging Face token here: ')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

import pandas as pd
df = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet")
print(df.shape)

Paste your Hugging Face token here: ··········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


(9841378, 31)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (inputs the model can use):
- `gsc_impressions` — search visibility signal
- `gsc_clicks` — click-through signal
- `gsc_avg_position` — ranking signal
- `ga4_engaged_sessions` — engagement signal
- `scroll_events` — on-page behavior signal

**Label / proxy** (what we're trying to predict):
- Refresh-urgency score, derived from trend decline + staleness (not a raw column — built in Week 2)

**Context** (identifiers, not model inputs):
- `content_hash_id`, `client_hash_id`, `report_date`, `month`

**Excluded** (deliberately not used):
- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` — excluded because they're sparse (mostly NaN) and not needed for scoring refresh urgency; including them this early adds noise without helping the target.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three claims, each backed by a query:
1. **Grain** — one row = one `content_hash_id` + one `report_date`, no duplicates.
2. **Row count and date span** — the slice covers all of March 2026 (31 days), with 331,437 unique content items across 9.84M rows.
3. **Availability** — filtering with `IS TRUE` on `gsc_data_available`, only 36.69% of rows have real GSC data.

### Five features, with "available when?" for each

1. **gsc_impressions** — knowable at the decision moment because it's logged the same day search visibility occurs.
2. **gsc_clicks** — knowable at the decision moment because it's recorded the same day a user clicks through.
3. **gsc_avg_position** — knowable at the decision moment because ranking position is measured daily, not retroactively.
4. **ga4_engaged_sessions** — knowable at the decision moment because engagement is tracked in real time during the session.
5. **scroll_events** — knowable at the decision moment because scroll tracking fires during the same session.

### The leakage trap

I'll deliberately add a column derived from the label itself, show how it artificially inflates a quick score, then remove it and keep the honest result.

In [10]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# 1. Grain check
grain_check = df.groupby(['content_hash_id', 'report_date']).size()
print("Grain check — max rows per content_id+date combo:", grain_check.max())
print("Any duplicates?", (grain_check > 1).any())
print()

# 2. Row count + date span
print("Total rows:", len(df))
print("Unique content items:", df['content_hash_id'].nunique())
print("Date range:", df['report_date'].min(), "to", df['report_date'].max())
print()

# 3. Availability check
available = df[df['gsc_data_available'] == True]
print("Rows with gsc_data_available == True:", len(available))
print("Percent available:", round(len(available) / len(df) * 100, 2), "%")
print()

# Five features frame
features_df = df[['content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks',
                   'gsc_avg_position', 'ga4_engaged_sessions', 'scroll_events']].copy()
print(features_df.head())
print()

# Leakage trap
sample = df[df['gsc_data_available'] == True].copy()
sample['engagement_rate'] = sample['ga4_engaged_sessions'] / sample['ga4_pageviews'].replace(0, np.nan)
sample['refresh_urgency'] = 1 - sample['engagement_rate'].fillna(0)

honest_features = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'scroll_events']
sample_clean = sample.dropna(subset=honest_features + ['refresh_urgency']).copy()

X_honest = sample_clean[honest_features]
y = sample_clean['refresh_urgency']
model = LinearRegression().fit(X_honest, y)
preds = model.predict(X_honest)
print("Honest MAE (no leakage):", mean_absolute_error(y, preds))

sample_clean['leaky_feature'] = sample_clean['refresh_urgency'] * 0.9 + np.random.normal(0, 0.01, len(sample_clean))
X_leaky = sample_clean[honest_features + ['leaky_feature']]
model_leaky = LinearRegression().fit(X_leaky, y)
preds_leaky = model_leaky.predict(X_leaky)
print("Leaky MAE (with label-derived column):", mean_absolute_error(y, preds_leaky))
print("Final honest MAE (leak removed, kept):", mean_absolute_error(y, preds))

Grain check — max rows per content_id+date combo: 1
Any duplicates? False

Total rows: 9841378
Unique content items: 331437
Date range: 2026-03-01 to 2026-03-31

Rows with gsc_data_available == True: 3611061
Percent available: 36.69 %

            content_hash_id report_date  gsc_impressions  gsc_clicks  \
0  content_b7e512995f79d5a6  2026-03-01               20           0   
1  content_05597932fe4da067  2026-03-01                1           0   
2  content_7a105f548d9c6916  2026-03-01              125           1   
3  content_905aa32a0230694e  2026-03-01                7           0   
4  content_a3ea9792f793ec72  2026-03-01               11           0   

   gsc_avg_position  ga4_engaged_sessions  scroll_events  
0          3.350000                   NaN            NaN  
1          0.000000                   NaN            NaN  
2          4.928000                   NaN            NaN  
3          4.000000                   NaN            NaN  
4          2.272727                 

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice cannot tell us *why* traffic changed — only that it did. Any refresh-urgency score built from it is an observed, directional signal, not a causal explanation, and should support prioritization decisions rather than prove a refresh will fix the underlying drop.

The data also carries a meaningful availability gap: only 36.69% of rows have `gsc_data_available == True`, meaning search performance data is missing or unreliable for most rows in this slice. GA4-derived fields (`ga4_engaged_sessions`, `scroll_events`) show substantial missingness too, likely from inconsistent GA4 rollout across clients rather than genuine zero engagement — treating these as true zeros would bias any downstream model.

Finally, this is a single-month window (March 2026). It cannot capture seasonality, multi-month trend shifts, or longer-term content lifecycle patterns — a page that appears stable this month may behave very differently across a full year, so conclusions here are a snapshot, not a generalizable pattern.

In [11]:
print("Rows missing GSC data:", (df['gsc_data_available'] != True).sum())
print("Percent missing GSC data:", round((df['gsc_data_available'] != True).sum() / len(df) * 100, 2), "%")
print()
print("GA4 availability:")
print(df['ga4_data_available'].value_counts(dropna=False))

Rows missing GSC data: 6230317
Percent missing GSC data: 63.31 %

GA4 availability:
ga4_data_available
False    6408671
None     3018741
True      413966
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.